In [1]:
import mne
import os.path as op
from pathlib import Path
import helpers.helper_functions as functions

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------

ss              = functions.settings_dict()
subject_idx_list = ss['subject_idx_list']

# Path to the fixed fsaverage source space.
# Must match the zooms=10 used in the original morph script.
fsaverage_src_file = op.join(
    ss['fs_subjects_dir'], "fsaverage", "bem", "fsaverage-vol-10-src.fif"
)

# Where to save the new morph files (keep separate from the old ones)
save_dir = Path(ss['morph_dir'])
save_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Create fsaverage source space if it doesn't exist yet
# ---------------------------------------------------------------------------

if not op.exists(fsaverage_src_file):
    print("Creating fsaverage volumetric source space (pos=10mm) ...")
    src_fsaverage = mne.setup_volume_source_space(
        "fsaverage",
        pos=10.0,                           # match original zooms=10
        subjects_dir=ss['fs_subjects_dir'],
        verbose=True,
    )
    Path(fsaverage_src_file).parent.mkdir(parents=True, exist_ok=True)
    src_fsaverage.save(fsaverage_src_file, overwrite=True)
    print(f"  Saved: {fsaverage_src_file}")
else:
    print(f"Loading existing fsaverage source space: {fsaverage_src_file}")

fsaverage_src = mne.read_source_spaces(fsaverage_src_file)
n_voxels_ref  = len(fsaverage_src[0]['vertno'])
print(f"  Reference voxels: {n_voxels_ref}")

# ---------------------------------------------------------------------------
# Recompute morphs with src_to
# ---------------------------------------------------------------------------

for ii in subject_idx_list:
    subject = ss['subject_id_list'][ii]
    print(f"\nProcessing {subject} ...")

    # -- Load forward solution (same special cases as original script) -------
    if ii == 1:
        # fwd_fname = op.join(ss['fwd_dir'], '_old', '070719', subject + '-fwd.fif')
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)
        fwd['src'][0]['subject_his_id'] = subject
    elif ii == 4:
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)
        fwd['src'][0]['subject_his_id'] = subject
    elif ii == 8:
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)
        fwd['src'][0]['src_mri_t']['trans'][2, 3] = -0.039
    else:
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)

    # -- Recompute morph with src_to -----------------------------------------
    morph = mne.compute_source_morph(
        fwd['src'],
        subject_from=subject,
        subject_to='fsaverage',
        src_to=fsaverage_src,               # <-- this is the key fix
        subjects_dir=ss['fs_subjects_dir'],
        zooms=10,                           # match original resolution
        verbose=True,
    )

    if ii == 8:
        morph.affine[2,3] = 170  # Correcting displacement in this dataset

    # -- Save new morph -------------------------------------------------------
    new_morph_fname = str(save_dir / f"{subject}-morph.h5")
    morph.save(new_morph_fname, overwrite=True)
    print(f"  Saved: {new_morph_fname}")

    # -- Quick sanity check: apply to a dummy STC and verify voxel count -----
    # (optional but useful to confirm all subjects now give the same count)
    dummy_data  = fwd['src'][0]['vertno']
    print(f"  Subject native voxels : {len(dummy_data)}")
    print(f"  Output will be pinned : {n_voxels_ref} fsaverage voxels")

print(f"\n{'='*60}")
print(f"Done. New morphs saved to: {save_dir}")
print("Update ss['morph_dir'] in your Z-map morphing script to point here,")
print("then re-run morph_zmap.py and permutation_ttest.py.")

Loading existing fsaverage source space: /media/elias/data/thesis_intermediate/data/scratch/fs_subjects_dir/fsaverage/bem/fsaverage-vol-10-src.fif
    Reading a source space...
    [done]
    1 source spaces read
  Reference voxels: 3031

Processing 0005_3SJ ...
Reading forward solution from /media/elias/data/thesis_intermediate/data/scratch/fwd/0005_3SJ-fwd.fif...
    Reading a source space...
    [done]
    1 source spaces read
    Desired named matrix (kind = 3523 (FIFF_MNE_FORWARD_SOLUTION_GRAD)) not available
    Read MEG forward solution (1440 sources, 306 channels, free orientations)
    Source spaces transformed to the forward solution coordinate frame
Volume source space(s) present...
    Loading /media/elias/data/thesis_intermediate/data/scratch/fs_subjects_dir/0005_3SJ/mri/brain.mgz as "from" volume
    Loading /media/elias/data/thesis_intermediate/data/scratch/fs_subjects_dir/fsaverage/mri/brain.mgz as "to" volume
Computing registration...
Reslicing to zooms=(10.0, 10.0, 10

In [ ]:
import mne
import os.path as op
import helpers.helper_functions as functions

# settings
ss = functions.settings_dict()

subject_idx_list = ss['subject_idx_list']
for ii in subject_idx_list:
    subject = ss['subject_id_list'][ii]

    # load forward. Replace conditioning by fixing fwd + dirs
    if ii == 1:
        fwd_fname = op.join(ss['fwd_dir'], '_old', '070719', subject + '-fwd.fif')  # original model
        fwd = mne.read_forward_solution(fwd_fname)
        fwd['src'][0]['subject_his_id'] = subject

    elif ii == 4:
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)
        fwd['src'][0]['subject_his_id'] = subject

    elif ii == 8:
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)
        fwd['src'][0]['src_mri_t']['trans'][2, 3] = -0.039  # src/mri mismatch corrected manually here.

    else:
        fwd_fname = op.join(ss['fwd_dir'], subject + '-fwd.fif')
        fwd = mne.read_forward_solution(fwd_fname)

    morph = mne.compute_source_morph(fwd['src'],
                                     subject_from=subject,
                                     subject_to='fsaverage',
                                     subjects_dir=ss['fs_subjects_dir'],
                                     zooms=10)

    #    if ii is 8: (moved to make_morph_img.py)
    ##        morph.affine[2,3] = 170  # Correcting displacement in this dataset
    #        morph.affine[2,3] = 149  # 10 mm res

    morph_fname = op.join(ss['scratch_dir'], 'morph', subject + '-morph.h5')
    morph.save(morph_fname, overwrite=True)